# Create the `arabin2026` RFI Channel Anomaly Dataset

This notebook creates a small fixed teaching package from the corrected RFI
artifacts. It keeps the ordinary supervised task binary (`BGN` versus
`NBRFI`) and exports a separate challenge of complete spectrograms containing
clean observations and more difficult RFI-containing observations.

The notebook writes a student-visible package and a supervisor-only directory.
It refuses to overwrite an existing export so that each package remains an
auditable snapshot.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'rfimt').is_dir():
            return candidate
    raise RuntimeError('Open this notebook from a directory inside the rfimt repository.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from rfimt.student_dataset import (
    STUDENT_CORE_LABELS,
    assert_disjoint_split_groups,
    assign_split_column,
    map_source_labels_to_student,
    select_challenge_segments,
    sha256_file,
    stack_complete_segments,
    student_readme,
)


## Read the declared source and output paths

The configuration is the single place where the source artifacts, challenge
size and export destination are declared. Changing a selection requires a new
configuration and output directory rather than silently changing a package.

In [2]:
CONFIG_PATH = REPO_ROOT / 'configs/student_datasets/b0531_rfi_channel_anomaly_student_v1.json'
with CONFIG_PATH.open(encoding='utf-8') as handle:
    spec = json.load(handle)

source = spec['source']
core_spec = spec['core']
challenge_spec = spec['challenge']
output_spec = spec['output']
export_root = Path(output_spec['root_directory'])
student_dir = export_root / output_spec['student_directory']
supervisor_dir = export_root / output_spec['supervisor_directory']

if export_root.exists():
    raise FileExistsError(f'Refusing to overwrite an existing student export: {export_root}')
print(f"Dataset: {spec['dataset_id']}")
print(f"Export: {export_root}")


Dataset: b0531_rfi_channel_anomaly_student_v1
Export: /hercules/results/akazantsev/rfim_dataset/student_exports/b0531_rfi_channel_anomaly_student_v1


## Load and audit the binary core data

The core array contains individual channel time series. The accompanying
metadata keeps the segment identity so that the three declared splits can be
checked for group overlap before any student-facing file is written.

In [3]:
core_meta = pd.read_csv(source['core_metadata_path'], keep_default_na=False)
core_series = np.load(source['core_array_path'], mmap_mode='r')
with np.load(source['core_split_indices_path']) as split_file:
    core_splits = {name: np.asarray(split_file[name], dtype=int) for name in ('train', 'val', 'test')}

if len(core_meta) != len(core_series):
    raise ValueError('Core metadata and channel-time-series array have different row counts.')
if tuple(core_series.shape[1:]) != (core_spec['segment_shape'][1],):
    raise ValueError(f"Expected one {core_spec['segment_shape'][1]}-sample channel series, got {core_series.shape}.")
if set(core_meta['label'].astype(str).unique()) != set(core_spec['source_labels']):
    raise ValueError('Core source must contain exactly the declared source labels.')
# The student-facing BGN name prevents the source label None from looking like a missing value.
core_meta['label'] = map_source_labels_to_student(core_meta['label'])
if set(core_meta['label'].unique()) != set(core_spec['labels']):
    raise ValueError('Student label mapping does not match the declared binary labels.')
for column in ('source_row_index', core_spec['group_column'], core_spec['channel_column']):
    if column not in core_meta:
        raise ValueError(f'Missing required core metadata column: {column}')

core_meta = assign_split_column(core_meta, core_splits)
assert_disjoint_split_groups(core_meta, group_col=core_spec['group_column'])
core_meta['binary_target'] = core_meta['label'].eq('NBRFI').astype(int)
print(core_meta.groupby(['split', 'label']).size().rename('channel_rows'))
print(core_meta.groupby('split')[core_spec['group_column']].nunique().rename('segments'))


split  label
test   BGN      1000
       NBRFI    1000
train  BGN      8000
       NBRFI    8000
val    BGN      1000
       NBRFI    1000
Name: channel_rows, dtype: int64
split
test      1242
train    10001
val       1243
Name: segments, dtype: int64


## Write the student-visible core

The compact core contains no duplicated views of the data. The metadata tells
the student which individual channel time series belongs to each split and how
it maps back to a complete segment.

In [4]:
export_root.mkdir(parents=True)
student_dir.mkdir()
supervisor_dir.mkdir()

core_output = np.asarray(core_series, dtype=np.float32)
core_table = core_meta.copy()
core_table.insert(0, 'core_example_index', np.arange(len(core_table), dtype=int))
core_columns = [
    'core_example_index', 'source_row_index', core_spec['group_column'],
    core_spec['channel_column'], 'frequency', 'split', 'label', 'binary_target',
]
missing_public_columns = [column for column in core_columns if column not in core_table]
if missing_public_columns:
    raise ValueError(f'Cannot build core metadata; missing columns: {missing_public_columns}')

np.save(student_dir / 'core_channel_time_series.npy', core_output)
core_table[core_columns].to_csv(student_dir / 'core_channel_metadata.csv', index=False)


## Select and export the reality challenge

Challenge spectra are complete 256 by 256 dynamic spectra. They are never used
for training, validation or threshold choice. The student receives no channel
labels for them; the supervisor keeps the reference annotations separately.

In [5]:
full_meta = pd.read_csv(source['full_metadata_path'], keep_default_na=False).copy()
full_meta['source_row_index'] = np.arange(len(full_meta), dtype=int)
full_array = np.load(source['full_array_path'], mmap_mode='r')
if len(full_meta) != len(full_array):
    raise ValueError('Full metadata and full channel-time-series array have different row counts.')
core_source_indices = core_meta['source_row_index'].to_numpy(dtype=int)
if core_source_indices.min() < 0 or core_source_indices.max() >= len(full_array):
    raise ValueError('Core source_row_index falls outside the full array.')
if not np.array_equal(np.asarray(core_series), np.asarray(full_array[core_source_indices])):
    raise ValueError('Core array does not match the declared source rows in the full array.')

challenge_selection = select_challenge_segments(
    full_meta,
    excluded_segments=core_meta[core_spec['group_column']].unique() if challenge_spec['exclude_core_segments'] else (),
    n_clean=int(challenge_spec['clean_segments']),
    n_hard=int(challenge_spec['rfi_containing_segments']),
    random_state=int(challenge_spec['random_state']),
    group_col=core_spec['group_column'],
)
challenge_spectra = stack_complete_segments(
    full_array,
    full_meta,
    challenge_selection[core_spec['group_column']].tolist(),
    group_col=core_spec['group_column'],
    channel_col=core_spec['channel_column'],
    expected_channels=core_spec['segment_shape'][0],
    expected_time=core_spec['segment_shape'][1],
)
challenge_overlaps_core = bool(
    set(challenge_selection[core_spec['group_column']]) & set(core_meta[core_spec['group_column']])
)
if challenge_spec['exclude_core_segments'] and challenge_overlaps_core:
    raise RuntimeError('Challenge and core data share at least one segment despite the declared exclusion.')

np.save(student_dir / 'challenge_spectrogram_segments.npy', challenge_spectra)
student_challenge_meta = challenge_selection[[core_spec['group_column']]].copy()
student_challenge_meta.insert(0, 'challenge_segment_index', np.arange(len(student_challenge_meta), dtype=int))
student_challenge_meta.to_csv(student_dir / 'challenge_segment_metadata.csv', index=False)
print(challenge_selection['challenge_kind'].value_counts())


challenge_kind
clean             12
rfi_containing    12
Name: count, dtype: int64


## Keep challenge annotations with the supervisor

The held-back table is intentionally separate from the student directory. It
allows a later honest comparison between the predicted mask and the original
channel-level annotations without exposing the answer during model selection.

In [6]:
supervisor_rows = []
for challenge_index, record in tqdm(challenge_selection.reset_index(drop=True).iterrows(), total=len(challenge_selection), desc='Writing supervisor labels', mininterval=5.0):
    segment_id = record[core_spec['group_column']]
    frame = full_meta.loc[full_meta[core_spec['group_column']].eq(segment_id)].sort_values(core_spec['channel_column'])
    for _, row in frame.iterrows():
        supervisor_rows.append({
            'challenge_segment_index': int(challenge_index),
            core_spec['group_column']: segment_id,
            core_spec['channel_column']: int(row[core_spec['channel_column']]),
            'source_label': str(row['label']),
            'binary_target': int(row['label'] == 'NBRFI'),
        })
supervisor_labels = pd.DataFrame(supervisor_rows)
supervisor_labels.to_csv(supervisor_dir / 'challenge_channel_labels.csv', index=False)
challenge_selection.to_csv(supervisor_dir / 'challenge_selection.csv', index=False)


Writing supervisor labels: 100%|██████████| 24/24 [00:00<00:00, 45.92it/s]


## Write documentation and manifests

The student manifest lists only files and information that belong in the
student package. The supervisor manifest adds the private challenge selection
and labels, making the later comparison reproducible without leaking them.

In [7]:
(student_dir / 'README.md').write_text(student_readme(), encoding='utf-8')
student_manifest = {
    'dataset_id': spec['dataset_id'],
    'array_semantics': {
        'core_channel_time_series.npy': ['core_example', 'time_sample'],
        'challenge_spectrogram_segments.npy': ['challenge_segment', 'frequency_channel', 'time_sample'],
    },
    'core_labels': list(STUDENT_CORE_LABELS),
    'source_to_student_label': {'None': 'BGN', 'NBRFI': 'NBRFI'},
    'core_split_row_counts': core_table['split'].value_counts().sort_index().to_dict(),
    'core_split_segment_counts': core_table.groupby('split')[core_spec['group_column']].nunique().to_dict(),
    'challenge_segment_count': int(len(challenge_selection)),
    'challenge_labels_included': False,
    'challenge_may_overlap_core_segments': not challenge_spec['exclude_core_segments'],
    'files': {},
}
for path in sorted(student_dir.iterdir()):
    if path.is_file() and path.name != 'student_manifest.json':
        student_manifest['files'][path.name] = {'sha256': sha256_file(path), 'bytes': path.stat().st_size}
with (student_dir / 'student_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(student_manifest, handle, indent=2, sort_keys=True)

supervisor_manifest = {
    'dataset_id': spec['dataset_id'],
    'challenge_selection_seed': int(challenge_spec['random_state']),
    'challenge_kind_counts': challenge_selection['challenge_kind'].value_counts().to_dict(),
    'student_manifest_sha256': sha256_file(student_dir / 'student_manifest.json'),
    'files': {path.name: {'sha256': sha256_file(path), 'bytes': path.stat().st_size} for path in sorted(supervisor_dir.iterdir())},
}
with (supervisor_dir / 'supervisor_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(supervisor_manifest, handle, indent=2, sort_keys=True)

print('Student package:', student_dir)
print('Supervisor-only annotations:', supervisor_dir)


Student package: /hercules/results/akazantsev/rfim_dataset/student_exports/b0531_rfi_channel_anomaly_student_v1/student_package
Supervisor-only annotations: /hercules/results/akazantsev/rfim_dataset/student_exports/b0531_rfi_channel_anomaly_student_v1/supervisor_only
